# Evaluacion de las respuestas generadas por LLM (RAGAS)

Evalua las 8 combinaciones de `resultados_generacion_llm.json` (2 pipelines de retrieve x 2 modelos x 2 estilos de prompt) sobre 4 dimensiones:

- **`faithfulness`**: ¿la respuesta se sostiene en el contexto recuperado, o inventa/extrapola mas alla de lo que dice el contexto? (mide alucinacion)
- **`answer_relevancy`** (`ResponseRelevancy`): ¿la respuesta realmente contesta lo que se pregunto, sin quedarse vaga o irse por la tangente?
- **`answer_correctness`**: ¿que tan cerca esta la respuesta generada de `respuesta_esperada`, combinando factualidad y similitud semantica?
- **`semantic_similarity`**: complemento barato y determinista de `answer_correctness` — solo similitud de embeddings entre respuesta y referencia, sin LLM de por medio (analogo a las metricas no-LLM que usamos para el retrieve).

**Resiliencia**: igual patron de 3 capas que en el notebook de evaluacion de retrieve (aqui SI aplica el fallback de modelos de OpenRouter para el LLM juez, porque cualquier juez razonable sirve — a diferencia del notebook de generacion, donde el modelo evaluado no podia sustituirse).

Este notebook ya incluye el parche para el import roto de `langchain_community.chat_models.vertexai` que nos encontramos la primera vez.

**Variables de entorno:** `OPENROUTER_API_KEY`.


In [ ]:
%pip install -q ragas langchain-openai pandas

In [1]:
import sys
import types

# Parche: ragas intenta importar ChatVertexAI de langchain_community aunque no
# lo usemos aqui. En versiones recientes de langchain_community ese modulo ya
# no existe, lo cual rompe el import de ragas por completo.
if "langchain_community.chat_models.vertexai" not in sys.modules:
    try:
        import langchain_community.chat_models.vertexai  # noqa: F401
    except ModuleNotFoundError:
        modulo_falso = types.ModuleType("langchain_community.chat_models.vertexai")

        class ChatVertexAI:  # stub, nunca se instancia de verdad
            pass

        modulo_falso.ChatVertexAI = ChatVertexAI
        sys.modules["langchain_community.chat_models.vertexai"] = modulo_falso
        print("Aviso: se aplico un parche al import roto de langchain_community.chat_models.vertexai.")

Aviso: se aplico un parche al import roto de langchain_community.chat_models.vertexai.


C:\Users\gomex\AppData\Local\Temp\ipykernel_52584\3143939404.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community.chat_models.vertexai  # noqa: F401


## 1. Cargar las combinaciones generadas


In [2]:
import json
from pathlib import Path

RUTA_GENERACION = "./validacion/resultados_generacion_llm.json"

with open(RUTA_GENERACION, "r", encoding="utf-8") as f:
    generacion = json.load(f)

combinaciones_generadas = generacion["combinaciones"]
print(f"Combinaciones cargadas: {len(combinaciones_generadas)}")
for c in combinaciones_generadas:
    etiqueta = f"{c['pipeline_retrieve']}__{c['modelo_llm']}__{c['estilo_prompt']}"
    print(f"  {etiqueta:55s} -> {c['total_preguntas']} respuestas")

Combinaciones cargadas: 8
  openai_cs800__claude-sonnet-5__directo                  -> 196 respuestas
  openai_cs800__claude-sonnet-5__estructurado             -> 196 respuestas
  openai_cs800__claude-haiku-4.5__directo                 -> 196 respuestas
  openai_cs800__claude-haiku-4.5__estructurado            -> 196 respuestas
  bge_m3_hibrido_cs800__claude-sonnet-5__directo          -> 196 respuestas
  bge_m3_hibrido_cs800__claude-sonnet-5__estructurado     -> 196 respuestas
  bge_m3_hibrido_cs800__claude-haiku-4.5__directo         -> 196 respuestas
  bge_m3_hibrido_cs800__claude-haiku-4.5__estructurado    -> 196 respuestas


## 2. LLM juez, embeddings evaluador y metricas


In [3]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("Define la variable de entorno OPENROUTER_API_KEY")

# Mismo juez economico que en la evaluacion de retrieve. Aqui SI tiene sentido
# el respaldo de modelos de OpenRouter: el juez no es la variable que se mide.
MODELO_JUEZ = "openai/gpt-4.1-mini"
MODELOS_JUEZ_RESPALDO = ["google/gemini-2.5-flash", "deepseek/deepseek-chat"]

llm_juez_base = ChatOpenAI(
    model=MODELO_JUEZ,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0,
    extra_body={"models": MODELOS_JUEZ_RESPALDO},
)
evaluator_llm = LangchainLLMWrapper(llm_juez_base)

# Embeddings para answer_correctness y semantic_similarity (mismo modelo que
# se uso para indexar, pero aqui es solo una herramienta de medicion, no parte
# de ningun pipeline de retrieve).
embeddings_base = OpenAIEmbeddings(
    model="openai/text-embedding-3-large",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings_base)

run_config = RunConfig(timeout=120, max_retries=8, max_wait=90, max_workers=4)

c:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\gomex\AppData\Local\Temp\ipykernel_52584\1087022036.py:23: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm_juez_base)
C:\Users\gomex\AppData\Local\Temp\ipykernel_52584\1087022036.py:33: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmb

In [5]:
from ragas.metrics import Faithfulness, ResponseRelevancy, AnswerCorrectness, SemanticSimilarity

METRICAS = [
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    AnswerCorrectness(llm=evaluator_llm, embeddings=evaluator_embeddings),
    SemanticSimilarity(embeddings=evaluator_embeddings),
]

print("Metricas a calcular:", [m.name for m in METRICAS])

Metricas a calcular: ['faithfulness', 'answer_relevancy', 'answer_correctness', 'semantic_similarity']


C:\Users\gomex\AppData\Local\Temp\ipykernel_52584\3686029345.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, AnswerCorrectness, SemanticSimilarity
C:\Users\gomex\AppData\Local\Temp\ipykernel_52584\3686029345.py:1: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy, AnswerCorrectness, SemanticSimilarity
C:\Users\gomex\AppData\Local\Temp\ipykernel_52584\3686029345.py:1: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. E

## 3. Evaluar cada combinacion (con checkpoint y reintentos)

None = evalua todas las preguntas de cada combinacion. Pon un numero chico (ej. 15) para una corrida de prueba rapida y barata antes de lanzar todo.


In [4]:
TAMANO_MUESTRA = None

RUTA_CHECKPOINTS = Path("./validacion/checkpoints_evaluacion_generacion")
RUTA_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

RUTA_SALIDA_DETALLE = "./validacion/evaluacion_generacion_detalle.csv"
RUTA_SALIDA_RESUMEN = "./validacion/evaluacion_generacion_resumen.csv"

In [6]:
from ragas import evaluate, EvaluationDataset


def construir_dataset(resultados, tamano_muestra=None):
    filas = resultados[:tamano_muestra] if tamano_muestra else resultados
    filas = [
        {
            "user_input": r["pregunta"],
            "response": r.get("respuesta_generada") or "",
            "retrieved_contexts": r.get("contextos_usados") or [],
            "reference": r.get("respuesta_esperada") or "",
        }
        for r in filas
    ]
    return EvaluationDataset.from_list(filas)

In [7]:
import time
import pandas as pd


def evaluar_combinacion(etiqueta, resultados, intentos_externos=3, espera_entre_intentos=60):
    ruta_checkpoint = RUTA_CHECKPOINTS / f"{etiqueta}.csv"

    if ruta_checkpoint.exists():
        print(f"  [{etiqueta}] ya evaluado antes, se reutiliza el checkpoint.")
        return pd.read_csv(ruta_checkpoint)

    dataset = construir_dataset(resultados, TAMANO_MUESTRA)

    for intento in range(1, intentos_externos + 1):
        try:
            resultado = evaluate(
                dataset=dataset,
                metrics=METRICAS,
                llm=evaluator_llm,
                embeddings=evaluator_embeddings,
                run_config=run_config,
                raise_exceptions=False,
                show_progress=True,
            )
            df = resultado.to_pandas()
            df.to_csv(ruta_checkpoint, index=False, encoding="utf-8")
            print(f"  [{etiqueta}] listo, checkpoint guardado en {ruta_checkpoint}")
            return df
        except Exception as e:
            print(f"  [{etiqueta}] intento {intento}/{intentos_externos} fallo: {e}")
            if intento < intentos_externos:
                print(f"  Esperando {espera_entre_intentos}s antes de reintentar la combinacion completa...")
                time.sleep(espera_entre_intentos)
            else:
                print(f"  [{etiqueta}] se agotaron los reintentos, se omite esta combinacion.")
                return None

In [8]:
resultados_por_combinacion = {}

for c in combinaciones_generadas:
    etiqueta = f"{c['pipeline_retrieve']}__{c['modelo_llm']}__{c['estilo_prompt']}"
    print(f"\n=== Evaluando: {etiqueta} ({c['total_preguntas']} preguntas) ===")
    df = evaluar_combinacion(etiqueta, c["resultados"])
    if df is not None:
        df["combinacion"] = etiqueta
        df["pipeline_retrieve"] = c["pipeline_retrieve"]
        df["modelo_llm"] = c["modelo_llm"]
        df["estilo_prompt"] = c["estilo_prompt"]
        resultados_por_combinacion[etiqueta] = df

print(f"\nCombinaciones evaluadas con exito: {len(resultados_por_combinacion)} / {len(combinaciones_generadas)}")


=== Evaluando: openai_cs800__claude-sonnet-5__directo (196 preguntas) ===


Evaluating: 100%|██████████| 784/784 [39:32<00:00,  3.03s/it]


  [openai_cs800__claude-sonnet-5__directo] listo, checkpoint guardado en validacion\checkpoints_evaluacion_generacion\openai_cs800__claude-sonnet-5__directo.csv

=== Evaluando: openai_cs800__claude-sonnet-5__estructurado (196 preguntas) ===


Evaluating: 100%|██████████| 784/784 [36:04<00:00,  2.76s/it]


  [openai_cs800__claude-sonnet-5__estructurado] listo, checkpoint guardado en validacion\checkpoints_evaluacion_generacion\openai_cs800__claude-sonnet-5__estructurado.csv

=== Evaluando: openai_cs800__claude-haiku-4.5__directo (196 preguntas) ===


Evaluating:  43%|████▎     | 334/784 [13:57<20:18,  2.71s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 784/784 [35:14<00:00,  2.70s/it]


... (7 outputs left out)

Evaluating: 100%|██████████| 784/784 [29:35<00:00,  2.27s/it]


  [bge_m3_hibrido_cs800__claude-haiku-4.5__directo] listo, checkpoint guardado en validacion\checkpoints_evaluacion_generacion\bge_m3_hibrido_cs800__claude-haiku-4.5__directo.csv

=== Evaluando: bge_m3_hibrido_cs800__claude-haiku-4.5__estructurado (196 preguntas) ===


Evaluating: 100%|██████████| 784/784 [29:57<00:00,  2.29s/it]

  [bge_m3_hibrido_cs800__claude-haiku-4.5__estructurado] listo, checkpoint guardado en validacion\checkpoints_evaluacion_generacion\bge_m3_hibrido_cs800__claude-haiku-4.5__estructurado.csv

Combinaciones evaluadas con exito: 8 / 8


In [9]:
if resultados_por_combinacion:
    df_detallado = pd.concat(resultados_por_combinacion.values(), ignore_index=True)
else:
    df_detallado = pd.DataFrame()

df_detallado.to_csv(RUTA_SALIDA_DETALLE, index=False, encoding="utf-8")
print(f"Detalle guardado en {RUTA_SALIDA_DETALLE} ({len(df_detallado)} filas)")

COLUMNAS_NO_METRICA = {
    "user_input", "response", "retrieved_contexts", "reference",
    "combinacion", "pipeline_retrieve", "modelo_llm", "estilo_prompt",
}
columnas_metricas = [c for c in df_detallado.columns if c not in COLUMNAS_NO_METRICA]

if not df_detallado.empty:
    resumen = (
        df_detallado.groupby(["combinacion", "pipeline_retrieve", "modelo_llm", "estilo_prompt"])[columnas_metricas]
        .mean()
        .reset_index()
        .sort_values("combinacion")
    )
    resumen.to_csv(RUTA_SALIDA_RESUMEN, index=False, encoding="utf-8")
    print(f"Resumen guardado en {RUTA_SALIDA_RESUMEN}")
else:
    resumen = pd.DataFrame()
    print("No hay resultados para resumir.")

resumen

Detalle guardado en ./validacion/evaluacion_generacion_detalle.csv (1568 filas)
Resumen guardado en ./validacion/evaluacion_generacion_resumen.csv


,combinacion,pipeline_retrieve,modelo_llm,estilo_prompt,faithfulness,answer_relevancy,answer_correctness,semantic_similarity
0,bge_m3_hibrido_cs800__claude-haiku-4.5__directo,bge_m3_hibrido_cs800,claude-haiku-4.5,directo,0.955692,0.755607,0.629282,0.774719
1,bge_m3_hibrido_cs800__claude-haiku-4.5__estruc...,bge_m3_hibrido_cs800,claude-haiku-4.5,estructurado,0.964258,0.759843,0.644004,0.790856
2,bge_m3_hibrido_cs800__claude-sonnet-5__directo,bge_m3_hibrido_cs800,claude-sonnet-5,directo,0.967930,0.723254,0.647584,0.798472
3,bge_m3_hibrido_cs800__claude-sonnet-5__estruct...,bge_m3_hibrido_cs800,claude-sonnet-5,estructurado,0.941525,0.722060,0.688077,0.801568
4,openai_cs800__claude-haiku-4.5__directo,openai_cs800,claude-haiku-4.5,directo,0.947827,0.764554,0.646647,0.781165
5,openai_cs800__claude-haiku-4.5__estructurado,openai_cs800,claude-haiku-4.5,estructurado,0.959307,0.773206,0.669052,0.799039
6,openai_cs800__claude-sonnet-5__directo,openai_cs800,claude-sonnet-5,directo,0.954659,0.720880,0.680518,0.800542
7,openai_cs800__claude-sonnet-5__estructurado,openai_cs800,claude-sonnet-5,estructurado,0.940759,0.700703,0.703045,0.802813


## Notas finales

- `evaluacion_generacion_detalle.csv` (una fila por pregunta x combinacion) y `evaluacion_generacion_resumen.csv` (promedio por combinacion) quedan en `./validacion/`.
- Costo/tiempo: cada fila dispara varias llamadas al LLM juez (faithfulness descompone la respuesta en afirmaciones y verifica cada una; answer_relevancy genera preguntas sinteticas) ademas de llamadas de embeddings — son mas llamadas por pregunta que en la evaluacion de retrieve. Corre primero con `TAMANO_MUESTRA = 15` para calibrar costo real antes de lanzar las 8 combinaciones completas.
- Si abres los CSV en Excel y los numeros salen en notacion cientifica rara, es el mismo tema de configuracion regional de siempre (separador decimal) — importa como texto/CSV especificando "." como decimal, no los abras con doble clic.
- Con esto ya tienes, para cada combinacion, las 4 metricas de generacion junto con las de retrieve del notebook anterior — se pueden cruzar por `pregunta` si quieres ver, por ejemplo, si las respuestas menos fieles coinciden con las preguntas donde el retrieve ya venia mas debil.
